# Ping: Direct vs Gateway - Protocol Comparison

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.4

def load_metrics(path):
    df = pd.read_json(path, lines=True)
    pts = df[df["type"] == "Point"].copy()
    exp = pd.json_normalize(pts["data"])
    exp["metric"] = pts["metric"].values
    exp["time"] = pd.to_datetime(exp["time"], utc=True)
    exp = exp.sort_values("time").reset_index(drop=True)
    exp["time_sec"] = (exp["time"] - exp["time"].min()).dt.total_seconds()
    return exp

## Configuration

Replace `TIMESTAMP` with the folder names from `../results/`.

In [ ]:
runs = {
    "direct / http":   load_metrics("../results/storage_direct_ping/http/smoke/TIMESTAMP/metrics.json"),
    "direct / https":  load_metrics("../results/storage_direct_ping/https/smoke/TIMESTAMP/metrics.json"),
    "direct / mtls":   load_metrics("../results/storage_direct_ping/mtls/smoke/TIMESTAMP/metrics.json"),
    "gateway / http":  load_metrics("../results/gateway_storage_ping/http/smoke/TIMESTAMP/metrics.json"),
    "gateway / https": load_metrics("../results/gateway_storage_ping/https/smoke/TIMESTAMP/metrics.json"),
    "gateway / mtls":  load_metrics("../results/gateway_storage_ping/mtls/smoke/TIMESTAMP/metrics.json"),
}

## Stats Summary

In [ ]:
rows = []
for label, df in runs.items():
    dur = df[df["metric"] == "http_req_duration"]["value"].dropna()
    rows.append({
        "run": label,
        "count": len(dur),
        "mean": round(dur.mean(), 2),
        "p50": round(np.percentile(dur, 50), 2),
        "p90": round(np.percentile(dur, 90), 2),
        "p95": round(np.percentile(dur, 95), 2),
        "p99": round(np.percentile(dur, 99), 2),
        "max": round(dur.max(), 2),
    })
pd.DataFrame(rows).set_index("run")

## Percentile Comparison

In [ ]:
pct_data = {}
for label, df in runs.items():
    dur = df[df["metric"] == "http_req_duration"]["value"].dropna()
    pct_data[label] = {
        "p50": np.percentile(dur, 50),
        "p90": np.percentile(dur, 90),
        "p95": np.percentile(dur, 95),
        "p99": np.percentile(dur, 99),
    }

ax = pd.DataFrame(pct_data).plot(kind="bar", figsize=(10, 5), rot=0)
ax.set_title("Ping Response Time Percentiles: Direct vs Gateway (ms)")
ax.set_ylabel("Duration (ms)")
ax.set_xlabel("Percentile")
for container in ax.containers:
    ax.bar_label(container, fmt="%.1f", fontsize=7, padding=2)
plt.tight_layout()
plt.show()

## Response Time Over Time

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#1f77b4", "#ff7f0e", "#2ca02c"]
styles = ["-", "-", "-", "--", "--", "--"]

for (label, df), color, style in zip(runs.items(), colors, styles):
    dur = df[df["metric"] == "http_req_duration"].set_index("time_sec")["value"]
    smoothed = dur.rolling(window=20, min_periods=1).mean()
    ax.plot(smoothed.index, smoothed.values, label=label, color=color, linestyle=style, linewidth=1.3, alpha=0.9)

ax.set_title("Response Time Over Time (ms, smoothed 20-sample) - solid: direct, dashed: gateway")
ax.set_xlabel("Elapsed time (s)")
ax.set_ylabel("Duration (ms)")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## TLS / Connection Timing Breakdown

In [ ]:
timing_metrics = [
    "http_req_blocked",
    "http_req_connecting",
    "http_req_tls_handshaking",
    "http_req_sending",
    "http_req_waiting",
    "http_req_receiving",
]
colors = ["#e07b39", "#5b9bd5", "#70ad47", "#ffc000", "#4472c4", "#ed7d31"]

timing_data = {}
for label, df in runs.items():
    timing_data[label] = {
        m: df[df["metric"] == m]["value"].dropna().mean() or 0
        for m in timing_metrics
    }

pd.DataFrame(timing_data).T.plot(
    kind="bar", stacked=True, figsize=(11, 5), color=colors, rot=25
)
plt.title("Mean Request Timing Breakdown (ms)")
plt.ylabel("Mean Duration (ms)")
plt.legend(loc="upper right", fontsize=8)
plt.tight_layout()
plt.show()